In [1]:
import os
import numpy as np
import pandas as pd
import gc
import itertools
import math
import torch
from itertools import product
from sklearn.model_selection import train_test_split


In [2]:
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/8"

train_path = os.path.join(SAMPLED_DATA_DIR, "hm_subset_train.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, "hm_subset_test.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# Cast IDs to str to handle mixed-type H&M IDs consistently
for df in [train_df, test_df]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

# Build contiguous 0-based encoders fitted on train only
user_enc = {u: i for i, u in enumerate(sorted(train_df["user_id"].unique()))}
item_enc = {v: i for i, v in enumerate(sorted(train_df["item_id"].unique()))}

def remap(df, user_enc, item_enc, drop_unknown=False):
    df = df.copy()
    if drop_unknown:
        df = df[df["user_id"].isin(user_enc) & df["item_id"].isin(item_enc)]
    df["user_id"] = df["user_id"].map(user_enc)
    df["item_id"] = df["item_id"].map(item_enc)
    return df.dropna(subset=["user_id", "item_id"]).astype({"user_id": int, "item_id": int})

train_df = remap(train_df, user_enc, item_enc)
test_df  = remap(test_df,  user_enc, item_enc, drop_unknown=True)

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)

n_users = len(user_enc)
n_items = len(item_enc)
print(f"Total Users : {n_users}")
print(f"Total Items : {n_items}")
train_df.head()


Total Users : 1760
Total Items : 2881


,user_id,item_id,bought
17812,23,2123,1
788,491,714,1
1399,357,1741,1
12060,590,1828,1
31118,1248,1025,1


In [3]:
# ── Build per-user train / val / test dicts ──────────────────────────────────
# Each dict maps uid -> np.array of shape (n, 3): [user_id, item_id, bought]
def df_to_dict(df):
    d = {}
    cols = list(df.columns[:3])   # user_id, item_id, bought
    for uid, grp in df.groupby(cols[0]):
        d[int(uid)] = grp[cols].values
    return d

train_dict = df_to_dict(train_df)
val_dict   = df_to_dict(val_df)
test_dict  = df_to_dict(test_df)

# Only keep users present in all three splits
common_users = sorted(set(train_dict) & set(val_dict) & set(test_dict))
train_dict = {u: train_dict[u] for u in common_users}
val_dict   = {u: val_dict[u]   for u in common_users}
test_dict  = {u: test_dict[u]  for u in common_users}

print(f"Active users (train∩val∩test): {len(common_users)}")
print(f"Train interactions : {sum(len(v) for v in train_dict.values())}")
print(f"Val   interactions : {sum(len(v) for v in val_dict.values())}")
print(f"Test  interactions : {sum(len(v) for v in test_dict.values())}")


Active users (train∩val∩test): 1757
Train interactions : 62516
Val   interactions : 15643
Test  interactions : 19541


In [7]:
# Sanity check: rating range (H&M 'bought' is binary 0/1)
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())
print(f"Rating range: [{min_rating}, {max_rating}]")
print(f"Column names : {list(train_df.columns)}")


Rating range: [1.0, 24.0]
Column names : ['user_id', 'item_id', 'bought']


In [9]:
# ── Local Learning MF ────────────────────────────────────────────────────────
# Each user trains a local MF model on their own data only.
# No communication. The item factors are private to each user.
# This is the worst-case baseline (lower bound on performance).

class LocalMF:
    """Single-user local matrix factorisation (no sharing).

    Parameters
    ----------
    n_items   : total item catalogue size (for consistent indexing)
    latent_dim: embedding dimension
    lr        : SGD learning rate
    reg       : L2 regularisation coefficient (applied to all factors)
    """
    def __init__(self, n_items, latent_dim=10, lr=0.01, reg=0.01):
        self.n_items    = n_items
        self.latent_dim = latent_dim
        self.lr         = lr
        self.reg        = reg
        # User has a single row vector; item factors are local copies
        self.u = np.random.normal(scale=0.1, size=latent_dim)
        self.Q = np.random.normal(scale=0.1, size=(n_items, latent_dim))  # item factors
        self.b = np.zeros(n_items)                                         # item biases

    def predict(self, item_id):
        return float(np.dot(self.u, self.Q[item_id]) + self.b[item_id])

    def train_epoch(self, interactions):
        """One SGD pass over this user's interactions.
        interactions: array of shape (n, 3) — [user_id, item_id, rating]
        """
        np.random.shuffle(interactions)
        mse = 0.0
        for row in interactions:
            item_id = int(row[1])
            rating  = float(row[2])
            pred    = self.predict(item_id)
            err     = rating - pred
            mse    += err ** 2
            # Gradients
            grad_u = -err * self.Q[item_id] + self.reg * self.u
            grad_q = -err * self.u           + self.reg * self.Q[item_id]
            grad_b = -err                    + self.reg * self.b[item_id]
            # Updates
            self.u          -= self.lr * grad_u
            self.Q[item_id] -= self.lr * grad_q
            self.b[item_id] -= self.lr * grad_b
        return np.sqrt(mse / len(interactions))

    def compute_rmse(self, interactions):
        mse = 0.0
        for row in interactions:
            item_id = int(row[1])
            rating  = float(row[2])
            mse    += (rating - self.predict(item_id)) ** 2
        return np.sqrt(mse / len(interactions))


In [11]:
# ── Training loop for all users ──────────────────────────────────────────────
import time

def run_local_learning(train_dict, val_dict, n_items,
                        latent_dim=10, lr=0.01, reg=0.01,
                        epochs=50, patience=5):
    """Train one LocalMF per user; return mean val RMSE per epoch and final models."""
    users = sorted(train_dict.keys())
    models = {u: LocalMF(n_items, latent_dim=latent_dim, lr=lr, reg=reg)
              for u in users}

    epoch_val_rmses = []
    epoch_times     = []
    best_val        = float('inf')
    no_improve      = 0

    for epoch in range(epochs):
        t0 = time.time()
        for u in users:
            models[u].train_epoch(train_dict[u].copy())  # copy so shuffle is local

        # Mean val RMSE across users
        val_rmses = [models[u].compute_rmse(val_dict[u]) for u in users]
        mean_val  = float(np.mean(val_rmses))
        epoch_val_rmses.append(mean_val)
        epoch_times.append(time.time() - t0)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | Val RMSE: {mean_val:.4f} | "
                  f"Time: {epoch_times[-1]:.1f}s")

        # Early stopping on val RMSE
        if mean_val < best_val:
            best_val   = mean_val
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    return models, epoch_val_rmses, epoch_times


In [13]:
# ── Grid search ───────────────────────────────────────────────────────────────
param_grid = {
    "latent_dim": [8, 10, 16],
    "lr":         [0.001, 0.005, 0.01],
    "reg":        [0.01, 0.05],
}

best_val_rmse = float("inf")
best_config   = None
gs_results    = []

for dim, lr, reg in product(param_grid["latent_dim"],
                             param_grid["lr"],
                             param_grid["reg"]):
    print(f"\nConfig: latent_dim={dim}, lr={lr}, reg={reg}")
    _, val_curve, _ = run_local_learning(
        train_dict, val_dict, n_items,
        latent_dim=dim, lr=lr, reg=reg,
        epochs=50, patience=5
    )
    final_val = val_curve[-1]
    gs_results.append((dim, lr, reg, final_val))
    print(f"  Final Val RMSE: {final_val:.4f}")
    if final_val < best_val_rmse:
        best_val_rmse = final_val
        best_config   = (dim, lr, reg)
    gc.collect()

print(f"\nBest config: latent_dim={best_config[0]}, lr={best_config[1]}, "
      f"reg={best_config[2]}, Val RMSE={best_val_rmse:.4f}")



Config: latent_dim=8, lr=0.001, reg=0.01
  Epoch  10 | Val RMSE: 1.6144 | Time: 0.4s
  Epoch  20 | Val RMSE: 1.6135 | Time: 0.4s
  Epoch  30 | Val RMSE: 1.6127 | Time: 0.4s
  Epoch  40 | Val RMSE: 1.6118 | Time: 0.4s
  Epoch  50 | Val RMSE: 1.6110 | Time: 0.4s
  Final Val RMSE: 1.6110

Config: latent_dim=8, lr=0.001, reg=0.05
  Epoch  10 | Val RMSE: 1.6145 | Time: 0.4s
  Epoch  20 | Val RMSE: 1.6136 | Time: 0.4s
  Epoch  30 | Val RMSE: 1.6128 | Time: 0.5s
  Epoch  40 | Val RMSE: 1.6120 | Time: 0.5s
  Epoch  50 | Val RMSE: 1.6112 | Time: 0.4s
  Final Val RMSE: 1.6112

Config: latent_dim=8, lr=0.005, reg=0.01
  Epoch  10 | Val RMSE: 1.6112 | Time: 0.6s
  Epoch  20 | Val RMSE: 1.6072 | Time: 0.5s
  Epoch  30 | Val RMSE: 1.6043 | Time: 0.4s
  Epoch  40 | Val RMSE: 1.6038 | Time: 0.5s
  Early stopping at epoch 42
  Final Val RMSE: 1.6040

Config: latent_dim=8, lr=0.005, reg=0.05
  Epoch  10 | Val RMSE: 1.6107 | Time: 0.4s
  Epoch  20 | Val RMSE: 1.6068 | Time: 0.4s
  Epoch  30 | Val RMSE: 

In [14]:
# ── Retrain best config, evaluate on test set ────────────────────────────────
print("Retraining best config on full train set ...")
best_models, val_curve, epoch_times = run_local_learning(
    train_dict, val_dict, n_items,
    latent_dim=best_config[0],
    lr=best_config[1],
    reg=best_config[2],
    epochs=50, patience=5
)

# Per-user test RMSE, then mean
test_rmses  = [best_models[u].compute_rmse(test_dict[u]) for u in sorted(best_models)]
mean_test   = float(np.mean(test_rmses))
print(f"\nTest RMSE (mean over users): {mean_test:.4f}")


Retraining best config on full train set ...
  Epoch  10 | Val RMSE: 1.6110 | Time: 0.4s
  Epoch  20 | Val RMSE: 1.6071 | Time: 0.4s
  Epoch  30 | Val RMSE: 1.6042 | Time: 0.4s
  Epoch  40 | Val RMSE: 1.6024 | Time: 0.5s
  Epoch  50 | Val RMSE: 1.6020 | Time: 0.4s

Test RMSE (mean over users): 1.6053


In [15]:
# ── Grid search results ──────────────────────────────────────────────────────
gs_df = pd.DataFrame(gs_results,
                     columns=["latent_dim", "lr", "reg", "val_rmse"])
print("Best config:")
print(gs_df.loc[gs_df["val_rmse"].idxmin()])
gs_df


Best config:
latent_dim    8.000000
lr            0.005000
reg           0.050000
val_rmse      1.601934
Name: 3, dtype: float64


,latent_dim,lr,reg,val_rmse
0,8,0.001,0.01,1.610984
1,8,0.001,0.05,1.611221
2,8,0.005,0.01,1.603957
3,8,0.005,0.05,1.601934
4,8,0.010,0.01,1.604674
5,8,0.010,0.05,1.602814
6,10,0.001,0.01,1.610340
7,10,0.001,0.05,1.611139
8,10,0.005,0.01,1.602266
9,10,0.005,0.05,1.602581


In [16]:
# ── Save per-epoch log ───────────────────────────────────────────────────────
log = {
    "epoch":    list(range(len(val_curve))),
    "val_rmse": val_curve,
    "time":     epoch_times,
}
log_df = pd.DataFrame(log)
log_df.to_csv("ll_hm.csv", index=False)
print(f"Saved ll_hm.csv  ({len(log_df)} epochs)")
log_df.head(10)


Saved ll_hm.csv  (50 epochs)


,epoch,val_rmse,time
0,0,1.614759,0.388806
1,1,1.614312,0.370655
2,2,1.613874,0.370635
3,3,1.613444,0.366451
4,4,1.613020,0.361674
5,5,1.612601,0.357650
6,6,1.612188,0.367348
7,7,1.611778,0.368729
8,8,1.611372,0.361753
9,9,1.610968,0.366546
